# JupyterLite（xeus-r）で学ぶ R 回帰分析 入門チュートリアル

このノートブックは、ブラウザだけで動く JupyterLite の R カーネル（xeus-r）で、
**回帰分析** の基本を一から学ぶチュートリアルです。
R 付属の `stats` パッケージと、統計モデリングの定番パッケージ **MASS**（このサイトの R 環境に組み込み済み）を使います。

## 対象者
- [R 統計検定入門](r_stats_tests_beginner_tutorial.ipynb) を終えた方
- 最小二乗法・ロジスティック回帰を「R でどう書くか」から学びたい方

## このチュートリアルで学ぶこと
1. 単回帰分析（`lm()` と `summary()` の読み方）
2. 重回帰分析
3. 回帰診断（残差プロット）
4. ダミー変数と交互作用
5. 予測と信頼区間・予測区間
6. ロジスティック回帰（`glm()`）
7. ポアソン回帰
8. MASS パッケージ：ロバスト回帰と負の二項回帰
9. モデル比較（AIC・分散分析）
10. まとめと総合演習

## JupyterLite で使ううえでの注意
- **セルの最後の式だけ** が自動表示されます。途中の結果は `print()` や `cat()` で出力します
- グラフ内の文字は **英語のみ**（日本語フォントが無いため）
- `install.packages()` は使えません。`MASS` はビルド時に組み込まれているので `library(MASS)` で読み込めます

---
## 0. 環境の準備

In [ ]:
Sys.setenv(TZ = "Asia/Tokyo")
options(repr.plot.width = 7, repr.plot.height = 4.5, repr.plot.res = 100, jupyter.plot_scale = 1)

cat("R のバージョン:", R.version.string, "\n")
cat("MASS が使えるか:", requireNamespace("MASS", quietly = TRUE), "\n")   # TRUE なら OK

---
## 1. 単回帰分析

回帰分析は「ある変数 y を、別の変数 x で説明・予測する」手法です。
まず、**教育年数から賃金を説明する** 単回帰を行います。
データは乱数で生成します（`set.seed()` で固定しているので結果は再現されます）。

$$ \text{wage} = \beta_0 + \beta_1 \times \text{education} + \varepsilon $$

In [ ]:
set.seed(10)
n <- 120
education  <- round(runif(n, 9, 18))       # 教育年数（9〜18 年）
experience <- round(runif(n, 0, 30))       # 実務経験年数
wage <- 80 + 22 * education + 9 * experience + rnorm(n, 0, 40)   # 月収（千円）
wage_data <- data.frame(wage = round(wage), education, experience)

print(head(wage_data))
summary(wage_data)

In [ ]:
model1 <- lm(wage ~ education, data = wage_data)   # 「wage を education で説明する」
summary(model1)

### `summary()` の読み方

| 項目 | 意味 |
|---|---|
| `Estimate` | 回帰係数の推定値。`education` の係数は「教育年数が 1 年増えると賃金が何千円増えるか」 |
| `Std. Error` | 係数の標準誤差（推定のばらつき） |
| `t value` / `Pr(>|t|)` | 「係数 = 0」の t 検定。p < 0.05 なら「効果あり」と判断 |
| `Residual standard error` | 残差の標準偏差（当てはめの誤差の大きさ） |
| `Multiple R-squared` | 決定係数。y のばらつきのうちモデルで説明できた割合（0〜1） |
| `F-statistic` | 「すべての係数 = 0」の検定。モデル全体が意味を持つか |

切片（`(Intercept)`）は「education = 0 のときの賃金」で、データの範囲外なので解釈しすぎないことも大切です。

In [ ]:
plot(wage_data$education, wage_data$wage, main = "Wage vs education",
     xlab = "Education (years)", ylab = "Wage (thousand yen)", pch = 19, col = "steelblue")
abline(model1, col = "red", lwd = 2)   # 回帰直線を重ねる

In [ ]:
print(coef(model1))       # 係数だけ取り出す
print(confint(model1))    # 係数の 95% 信頼区間
cat("決定係数 R^2:", round(summary(model1)$r.squared, 3), "\n")

### 練習問題 1

8 章までで使う広告費と売上のデータを作って単回帰してみましょう。
`set.seed(21); ad <- runif(50, 5, 50); sales <- 200 + 6 * ad + rnorm(50, 0, 40)`
1. `lm()` で sales を ad に回帰し、`summary()` を表示する
2. 散布図に回帰直線を重ねる（ラベルは英語）
3. 「広告費を 1 万円増やすと売上は平均いくら増えるか」をコメントで答える

In [ ]:
# 練習問題 1 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 1 の解答例を見る</strong></summary>

```r
set.seed(21)
ad <- runif(50, 5, 50)
sales <- 200 + 6 * ad + rnorm(50, 0, 40)

ad_model <- lm(sales ~ ad)
print(summary(ad_model))

plot(ad, sales, main = "Sales vs ad spend", xlab = "Ad spend", ylab = "Sales",
     pch = 19, col = "steelblue")
abline(ad_model, col = "red", lwd = 2)

# ad の係数（約 6）が「広告費 1 単位あたりの売上の平均増加」。
```

</details>

---
## 2. 重回帰分析

説明変数を複数使うには、式を `+` でつなぎます。
`wage ~ education + experience` は「教育年数と経験年数の両方で賃金を説明する」という意味です。

重回帰の係数は「**他の変数を一定としたときの** 効果」と解釈します。

In [ ]:
model2 <- lm(wage ~ education + experience, data = wage_data)
summary(model2)

単回帰（R² 約 0.5）に比べて、決定係数が大きく上がったはずです。
説明変数の数が違うモデル同士は、変数の数を罰則にした **自由度調整済み R²**（`Adjusted R-squared`）で比べます。

### 多重共線性のチェック

説明変数同士の相関が非常に高い（目安 0.8〜0.9 超）と、係数の推定が不安定になります（多重共線性）。
まず相関行列で確認しましょう。

In [ ]:
print(round(cor(wage_data[, c("education", "experience")]), 3))
# 今回はほぼ無相関なので問題なし。
# 相関が非常に高い変数があるときは、片方を落とす・合成するなどを検討する。

### 練習問題 2

`model2` の結果から、
1. 「経験 1 年の効果」は何千円か、95% 信頼区間付きで示してください（`confint()`）
2. 教育 16 年・経験 10 年の人の賃金の予測値を、係数から**手計算**（`coef(model2)` を使った式）で求めてください

In [ ]:
# 練習問題 2 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 2 の解答例を見る</strong></summary>

```r
print(coef(model2))
print(confint(model2))

b <- coef(model2)
pred_manual <- b[1] + b[2] * 16 + b[3] * 10
cat("教育 16 年・経験 10 年の予測賃金:", round(unname(pred_manual), 1), "千円\n")
```

</details>

---
## 3. 回帰診断（残差プロット）

回帰分析の仮定（直線性・等分散性・正規性・外れ値の影響）は、**残差** を図で確認します。
`plot(モデル)` で代表的な 4 つの診断プロットが描けます。

| プロット | 見るポイント |
|---|---|
| Residuals vs Fitted | 残差に曲がったパターンがないか（あれば非線形） |
| Q-Q Residuals | 残差が直線に乗るか（正規性） |
| Scale-Location | 残差の大きさが予測値によって変わらないか（等分散性） |
| Residuals vs Leverage | 影響の大きい外れ値（クックの距離が大きい点）がないか |

In [ ]:
par(mfrow = c(2, 2))   # 2×2 に並べて表示
plot(model2)
par(mfrow = c(1, 1))

In [ ]:
# 残差のヒストグラムでも正規性を確認できる
res2 <- residuals(model2)
hist(res2, main = "Residuals of model2", xlab = "Residual", col = "gray")
cat("残差の平均（ほぼ 0 になる）:", round(mean(res2), 6), "\n")

---
## 4. ダミー変数と交互作用

### 4.1 質的変数（factor）を説明変数にする

業種（Manufacturing / Service / IT）のような質的変数は、`factor` にして式に入れるだけで
自動的に **ダミー変数** に展開されます。

In [ ]:
set.seed(11)
sector <- factor(sample(c("Manufacturing", "Service", "IT"), n, replace = TRUE))
# IT は +60、Service は +20 の賃金プレミアムがあるデータを作る
wage_sector <- wage_data$wage + ifelse(sector == "IT", 60, ifelse(sector == "Service", 20, 0))
sector_data <- data.frame(wage = wage_sector, education, experience, sector)

print(table(sector_data$sector))
model3 <- lm(wage ~ education + experience + sector, data = sector_data)
summary(model3)

係数 `sectorManufacturing` や `sectorService` は、**基準カテゴリ**（既定ではアルファベット順で最初の `IT`）
との賃金差を表します。基準を変えるには `relevel()` を使います。

In [ ]:
sector_data$sector <- relevel(sector_data$sector, ref = "Manufacturing")   # 基準を製造業に
model3b <- lm(wage ~ education + experience + sector, data = sector_data)
print(round(coef(model3b), 1))
# sectorIT ≈ +60, sectorService ≈ +20 と、作ったとおりのプレミアムが推定される

### 4.2 交互作用

「教育の効果が業種によって違う」ような **組み合わせの効果** は交互作用項で表します。
- `x1:x2` … 交互作用項のみ
- `x1 * x2` … 主効果 + 交互作用（`x1 + x2 + x1:x2` の省略形）

In [ ]:
model4 <- lm(wage ~ education * sector, data = sector_data)
print(round(coef(summary(model4)), 3))
# education:sectorXX が有意でなければ「教育の効果は業種で変わらない」と判断できる

### 練習問題 3

1 章の演習で作った広告データに「チャネル」を追加します。
`set.seed(22); channel <- factor(sample(c("TV", "Web"), 50, replace = TRUE))`、
`sales2 <- sales + ifelse(channel == "Web", 50, 0)`
1. `sales2 ~ ad + channel` の重回帰を行い、Web チャネルのプレミアムが約 50 と推定されることを確認する
2. `sales2 ~ ad * channel` で交互作用を入れ、「広告の効果はチャネルで違うか」を確認する

In [ ]:
# 練習問題 3 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 3 の解答例を見る</strong></summary>

```r
set.seed(22)
channel <- factor(sample(c("TV", "Web"), 50, replace = TRUE))
sales2 <- sales + ifelse(channel == "Web", 50, 0)

m_ch <- lm(sales2 ~ ad + channel)
print(summary(m_ch))   # channelWeb ≈ 50

m_int <- lm(sales2 ~ ad * channel)
print(round(coef(summary(m_int)), 3))
# ad:channelWeb の p 値が大きければ「広告の効果はチャネルで同じ」と判断できる
```

</details>

---
## 5. 予測と信頼区間・予測区間

作ったモデルで新しいデータを予測するには `predict(モデル, newdata = ...)` を使います。

- `interval = "confidence"` … **平均の** 信頼区間（回帰直線の位置の不確かさ）
- `interval = "prediction"` … **個々の値の** 予測区間（個人差の分も広くなる）

In [ ]:
new_person <- data.frame(education = c(12, 16), experience = c(5, 10))
print(predict(model2, newdata = new_person))
cat("\n95% 信頼区間（平均の区間）:\n")
print(round(predict(model2, newdata = new_person, interval = "confidence"), 1))
cat("\n95% 予測区間（個々の値の区間。より広い）:\n")
print(round(predict(model2, newdata = new_person, interval = "prediction"), 1))

In [ ]:
# 経験年数を平均に固定して、教育年数と賃金の関係を区間付きで図示する
edu_grid <- data.frame(education = seq(9, 18, by = 0.5),
                       experience = mean(wage_data$experience))
ci <- predict(model2, newdata = edu_grid, interval = "confidence")
pi <- predict(model2, newdata = edu_grid, interval = "prediction")

plot(wage_data$education, wage_data$wage, pch = 19, col = "gray",
     main = "Confidence vs prediction interval",
     xlab = "Education (years)", ylab = "Wage (thousand yen)")
lines(edu_grid$education, ci[, "fit"], col = "red", lwd = 2)
matlines(edu_grid$education, ci[, c("lwr", "upr")], col = "red", lty = 2)
matlines(edu_grid$education, pi[, c("lwr", "upr")], col = "blue", lty = 3)
legend("topleft", legend = c("Fit", "95% confidence", "95% prediction"),
       col = c("red", "red", "blue"), lty = c(1, 2, 3), lwd = c(2, 1, 1))

### 練習問題 4

1 章の演習の広告モデル（`ad_model`）で、広告費 10・25・40 のときの売上を
1. 点予測
2. 95% 予測区間付き
で求めてください。

In [ ]:
# 練習問題 4 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 4 の解答例を見る</strong></summary>

```r
new_ad <- data.frame(ad = c(10, 25, 40))
print(predict(ad_model, newdata = new_ad))
print(round(predict(ad_model, newdata = new_ad, interval = "prediction"), 1))
```

</details>

---
## 6. ロジスティック回帰

y が **0 / 1**（購入する・しない、合格・不合格）のときは、`glm(..., family = binomial)` で
ロジスティック回帰を使います。確率 p を次の形でモデル化します。

$$ \log\frac{p}{1-p} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 $$

左辺は **対数オッズ** と呼ばれ、係数の指数 `exp(係数)` が **オッズ比**
（その変数が 1 増えたときにオッズが何倍になるか）になります。

In [ ]:
set.seed(12)
n2 <- 300
income <- round(rnorm(n2, 500, 120))   # 年収（万円）
age    <- round(runif(n2, 20, 69))     # 年齢
z <- -6 + 0.008 * income + 0.04 * age  # 真のモデル（対数オッズ）
buy <- rbinom(n2, 1, 1 / (1 + exp(-z)))   # 1 = 商品を購入

buy_data <- data.frame(buy, income, age)
print(head(buy_data))
cat("全体の購入率:", round(mean(buy_data$buy), 3), "\n")

In [ ]:
logit1 <- glm(buy ~ income + age, data = buy_data, family = binomial)
summary(logit1)

In [ ]:
# オッズ比とその信頼区間（Wald 法: confint.default）
print(round(exp(coef(logit1)), 4))
print(round(exp(confint.default(logit1)), 4))
# income のオッズ比 1.008 → 年収が 1 万円増えると購入オッズが 0.8% 増える
# 100 万円増なら 1.008^100 ≈ 2.2 倍

In [ ]:
# 予測確率と的中率（確率 0.5 以上を「購入する」と予測）
pred_prob  <- predict(logit1, type = "response")   # type = "response" で確率を返す
pred_class <- ifelse(pred_prob >= 0.5, 1, 0)

print(table(actual = buy_data$buy, predicted = pred_class))
cat("的中率:", round(mean(pred_class == buy_data$buy), 3), "\n")

# 新しい顧客の購入確率
new_cust <- data.frame(income = c(300, 500, 800), age = c(25, 40, 60))
print(round(predict(logit1, newdata = new_cust, type = "response"), 3))

### 練習問題 5

勉強時間と模試の合否データを作って分析しましょう。
`set.seed(23); hours <- runif(200, 0, 20); pass <- rbinom(200, 1, 1 / (1 + exp(-(-4 + 0.5 * hours))))`
1. ロジスティック回帰で合格確率を勉強時間で説明する
2. 勉強時間のオッズ比を求め、意味をコメントで説明する
3. 勉強時間 5・10・15 時間の合格確率を予測する

In [ ]:
# 練習問題 5 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 5 の解答例を見る</strong></summary>

```r
set.seed(23)
hours <- runif(200, 0, 20)
pass <- rbinom(200, 1, 1 / (1 + exp(-(-4 + 0.5 * hours))))

logit_pass <- glm(pass ~ hours, family = binomial)
print(summary(logit_pass))

cat("オッズ比:", round(exp(coef(logit_pass)["hours"]), 3), "\n")
# 勉強時間が 1 時間増えるごとに合格オッズが約 1.6 倍になる

print(round(predict(logit_pass, newdata = data.frame(hours = c(5, 10, 15)),
                    type = "response"), 3))
```

</details>

---
## 7. ポアソン回帰

y が **回数（0, 1, 2, ...）** のカウントデータのときは、`glm(..., family = poisson)` を使います。
係数の指数は「その変数が 1 増えたときに平均回数が何倍になるか」（発生率比）です。

In [ ]:
set.seed(13)
n3 <- 250
member_year <- round(runif(n3, 0, 10))   # 会員歴（年）
coupon <- rbinom(n3, 1, 0.4)             # クーポン利用 (1 = あり)
lambda <- exp(0.3 + 0.12 * member_year + 0.5 * coupon)   # 真の平均来店回数
visits <- rpois(n3, lambda)              # 月間来店回数

visit_data <- data.frame(visits, member_year, coupon)
print(head(visit_data))
pois1 <- glm(visits ~ member_year + coupon, data = visit_data, family = poisson)
summary(pois1)

In [ ]:
print(round(exp(coef(pois1)), 3))   # 発生率比: クーポン利用者は来店回数が約 1.65 倍
cat("過分散の目安 (residual deviance / df):",
    round(pois1$deviance / pois1$df.residual, 2), "\n")
# この値が 1 に近ければポアソン分布の仮定は妥当。
# 1 を大きく超える（目安 1.5〜2 以上）なら「過分散」→ 8 章の負の二項回帰へ

---
## 8. MASS パッケージ：ロバスト回帰と負の二項回帰

ここからは `MASS` パッケージを使います（このサイトの R 環境に組み込み済み）。

In [ ]:
library(MASS)
cat("MASS を読み込みました\n")

### 8.1 ロバスト回帰 `rlm()`

最小二乗法（`lm`）は外れ値に大きく引っ張られます。
`rlm()`（M 推定）は外れ値の影響を自動的に抑えます。

In [ ]:
set.seed(14)
x_out <- runif(60, 0, 10)
y_out <- 5 + 2 * x_out + rnorm(60, 0, 2)   # 真の関係: y = 5 + 2x
y_out[c(5, 12, 20)] <- y_out[c(5, 12, 20)] + 40   # 3 点だけ大きな外れ値を混入

ols_fit <- lm(y_out ~ x_out)
rob_fit <- rlm(y_out ~ x_out)

cat("真の係数        : 切片 5, 傾き 2\n")
cat("lm  (最小二乗法):", round(coef(ols_fit), 2), "\n")
cat("rlm (ロバスト)  :", round(coef(rob_fit), 2), "\n")

In [ ]:
plot(x_out, y_out, pch = 19, col = "gray40", main = "OLS vs robust regression",
     xlab = "x", ylab = "y")
abline(ols_fit, col = "red", lwd = 2)
abline(rob_fit, col = "blue", lwd = 2, lty = 2)
abline(a = 5, b = 2, col = "darkgreen", lwd = 1, lty = 3)
legend("topleft", legend = c("lm (OLS)", "rlm (robust)", "True line"),
       col = c("red", "blue", "darkgreen"), lty = c(1, 2, 3), lwd = c(2, 2, 1))

### 8.2 負の二項回帰 `glm.nb()`

カウントデータの分散が平均より大きい（**過分散**）ときは、ポアソン回帰の代わりに
負の二項回帰を使うと標準誤差を正しく評価できます。

In [ ]:
set.seed(15)
# 過分散のある来店回数データ（同じ平均構造 + 個人差によるばらつき）
visit_data$visits_od <- rnbinom(n3, size = 1.5, mu = lambda)

pois_bad <- glm(visits_od ~ member_year + coupon, data = visit_data, family = poisson)
cat("ポアソンの過分散指標:", round(pois_bad$deviance / pois_bad$df.residual, 2), "  ← 1 を大きく超える\n")

nb_fit <- glm.nb(visits_od ~ member_year + coupon, data = visit_data)
summary(nb_fit)

### 練習問題 6

`set.seed(24); xr <- runif(40, 0, 10); yr <- 10 + 3 * xr + rnorm(40, 0, 3); yr[c(3, 8)] <- yr[c(3, 8)] - 50`
という外れ値入りデータで、
1. `lm()` と `rlm()` の係数を比べてください（真の傾きは 3）
2. 散布図に両方の回帰直線を重ねてください

In [ ]:
# 練習問題 6 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 6 の解答例を見る</strong></summary>

```r
set.seed(24)
xr <- runif(40, 0, 10)
yr <- 10 + 3 * xr + rnorm(40, 0, 3)
yr[c(3, 8)] <- yr[c(3, 8)] - 50

fit_ols <- lm(yr ~ xr)
fit_rob <- rlm(yr ~ xr)
cat("lm :", round(coef(fit_ols), 2), "\n")
cat("rlm:", round(coef(fit_rob), 2), "\n")   # rlm のほうが傾き 3 に近い

plot(xr, yr, pch = 19, col = "gray40", main = "OLS vs robust", xlab = "x", ylab = "y")
abline(fit_ols, col = "red", lwd = 2)
abline(fit_rob, col = "blue", lwd = 2, lty = 2)
legend("topleft", legend = c("lm", "rlm"), col = c("red", "blue"), lty = c(1, 2), lwd = 2)
```

</details>

---
## 9. モデル比較

### 9.1 AIC（赤池情報量規準）

AIC は「当てはまりの良さ」と「モデルの複雑さ」のバランスを測る指標で、**小さいほど良い** モデルです。
同じデータ・同じ y に当てはめたモデル同士で比べます。

In [ ]:
model2q <- lm(wage ~ education + experience + I(experience^2), data = wage_data)  # 2 乗項を追加
print(AIC(model1, model2, model2q))
# model2 が最小 → 2 乗項を足しても改善しない（データは線形に作ったので妥当な結果）

### 9.2 入れ子モデルの F 検定

「変数を追加して意味があったか」は、`anova(小さいモデル, 大きいモデル)` の F 検定で調べられます。

In [ ]:
anova(model1, model2)   # experience を追加した効果は有意か

In [ ]:
# step(): AIC を基準に変数を自動で取捨選択する
step_fit <- step(model2q, trace = 0)   # trace = 0 で途中経過を表示しない
print(coef(step_fit))
# I(experience^2) が自動的に落とされ、model2 と同じ形に戻る

---
## 10. まとめ

| 目的 | y の型 | 関数 |
|---|---|---|
| 直線関係のモデル化 | 連続値 | `lm(y ~ x1 + x2)` |
| 質的変数・交互作用 | 連続値 | `lm(y ~ x * f)`（factor は自動でダミー化） |
| 予測 | — | `predict(model, newdata, interval =)` |
| 2 値の確率 | 0/1 | `glm(..., family = binomial)` → `exp(coef())` でオッズ比 |
| 回数 | カウント | `glm(..., family = poisson)`、過分散なら `MASS::glm.nb()` |
| 外れ値に頑健 | 連続値 | `MASS::rlm()` |
| モデル選択 | — | `AIC()`、`anova()`、`step()` |

**分析の流れ**: 散布図で関係を確認 → モデルを当てはめる → 診断プロットで仮定を確認 → 係数と区間を解釈 → 必要ならモデルを見直す

## 次のステップ
- [R 統計検定入門](r_stats_tests_beginner_tutorial.ipynb) — 検定の復習はこちら
- Python 側の [statsmodels 入門](../python/statsmodels/statsmodels_tutorial.ipynb) や [パネルデータ分析入門](../python/statsmodels/panel_data_beginner_tutorial.ipynb) で、同じ考え方を Python でも試せます

### 総合演習

中古マンションの価格データを作って、総合的に分析してください。

```r
set.seed(60)
m <- 150
area  <- round(runif(m, 40, 100))                      # 専有面積 (m2)
dist  <- round(runif(m, 1, 20))                        # 駅からの距離 (分)
aged  <- round(runif(m, 0, 40))                        # 築年数
price <- round(1000 + 45 * area - 60 * dist - 25 * aged + rnorm(m, 0, 300))  # 価格 (万円)
condo <- data.frame(price, area, dist, aged)
```

1. `price ~ area + dist + aged` の重回帰を行い、各係数の意味をコメントで説明する
2. 診断プロット（4 面）を描く
3. 面積 70 m²・駅 5 分・築 10 年の物件の予測価格を 95% 予測区間付きで求める
4. `price ~ area` だけの単回帰と AIC で比較する
5. 「価格が 4000 万円を超えるか」を 0/1 にして、ロジスティック回帰で面積・距離・築年数から説明する

In [ ]:
# 総合演習 の解答欄：ここにコードを書いてください

<details>
<summary><strong>総合演習 の解答例を見る</strong></summary>

```r
set.seed(60)
m <- 150
area  <- round(runif(m, 40, 100))
dist  <- round(runif(m, 1, 20))
aged  <- round(runif(m, 0, 40))
price <- round(1000 + 45 * area - 60 * dist - 25 * aged + rnorm(m, 0, 300))
condo <- data.frame(price, area, dist, aged)

# 1. 重回帰
full <- lm(price ~ area + dist + aged, data = condo)
print(summary(full))
# area: 1 m2 広いと約 +45 万円 / dist: 駅から 1 分遠いと約 -60 万円 / aged: 築 1 年で約 -25 万円

# 2. 診断プロット
par(mfrow = c(2, 2))
plot(full)
par(mfrow = c(1, 1))

# 3. 予測
new_room <- data.frame(area = 70, dist = 5, aged = 10)
print(round(predict(full, newdata = new_room, interval = "prediction"), 0))

# 4. AIC 比較
simple <- lm(price ~ area, data = condo)
print(AIC(simple, full))   # full のほうが小さい → 変数を増やす価値がある

# 5. ロジスティック回帰
condo$expensive <- ifelse(condo$price > 4000, 1, 0)
cat("4000 万円超の割合:", round(mean(condo$expensive), 3), "\n")
logit_c <- glm(expensive ~ area + dist + aged, data = condo, family = binomial)
print(summary(logit_c))
print(round(exp(coef(logit_c)), 3))   # オッズ比
```

</details>